# 02 — Commercial Cash Flow LGD Model

**PPSR + GSR Secured Lending — Australian Bank Practice**

This notebook demonstrates a commercial lending LGD framework covering:

| Step | Description |
|------|-------------|
| 1 | Load synthetic commercial default & workout data |
| 2 | EDA — LGD by security type, industry, coverage ratio |
| 3 | Segmentation — Security type × Coverage band × Industry |
| 4 | EAD with CCF for revolving/overdraft facilities |
| 5 | Recovery analysis by PPSR/GSR security type |
| 6 | Segmented LGD model with exposure-weighted long-run average |
| 7 | Downturn overlay (by security type) |
| 8 | MoC and supervisory LGD floor |
| 9 | SME firm-size adjustment |
| 10 | Validation & backtesting |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_commercial_data
from src.lgd_calculation import CommercialLGDEngine, exposure_weighted_average
from src.validation import (
    accuracy_metrics, discriminatory_power, conservatism_test,
    calibration_by_segment, compute_psi, generate_validation_report
)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

## 1. Generate & Load Data

In [ ]:
loans, cashflows = generate_commercial_data(n_loans=300, seed=43)

print(f"Loans: {loans.shape}")
print(f"Cashflows: {cashflows.shape}")
print(f"\nSecurity type distribution:")
display(loans['security_type'].value_counts())
print(f"\nFacility type distribution:")
display(loans['facility_type'].value_counts())
loans.head()

## 2. Exploratory Data Analysis

In [ ]:
print("=== Realised LGD Summary ===")
display(loans['realised_lgd'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LGD distribution
axes[0].hist(loans['realised_lgd'], bins=40, edgecolor='white', alpha=0.8, color='steelblue')
axes[0].axvline(loans['realised_lgd'].mean(), color='red', ls='--',
                label=f"Mean: {loans['realised_lgd'].mean():.2%}")
axes[0].set_title('Commercial Lending — Realised LGD Distribution')
axes[0].set_xlabel('LGD')
axes[0].legend()

# By security type
loans.boxplot(column='realised_lgd', by='security_type', ax=axes[1], rot=25)
axes[1].set_title('Realised LGD by Security Type')
axes[1].set_xlabel('Security Type')
axes[1].set_ylabel('Realised LGD')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'commercial_lgd_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Security coverage ratio vs LGD — primary risk driver
fig, ax = plt.subplots(figsize=(9, 6))
for sec_type in loans['security_type'].unique():
    mask = loans['security_type'] == sec_type
    ax.scatter(loans.loc[mask, 'security_coverage_ratio'], loans.loc[mask, 'realised_lgd'],
              alpha=0.5, s=20, label=sec_type)
ax.set_xlabel('Security Coverage Ratio')
ax.set_ylabel('Realised LGD')
ax.set_title('Security Coverage vs LGD by Security Type')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'commercial_coverage_vs_lgd.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LGD by key commercial drivers
print("=== LGD by Key Segments ===")
for col in ['security_type', 'seniority', 'facility_type', 'resolution_strategy', 'default_trigger']:
    print(f"\n--- {col} ---")
    summary = loans.groupby(col).agg(
        count=('realised_lgd', 'size'),
        mean_lgd=('realised_lgd', 'mean'),
        mean_ead=('ead', 'mean'),
        mean_coverage=('security_coverage_ratio', 'mean'),
    ).round(4)
    display(summary)

In [ ]:
# Top 10 industries by average LGD
industry_lgd = loans.groupby('industry').agg(
    count=('realised_lgd', 'size'),
    mean_lgd=('realised_lgd', 'mean'),
    total_ead=('ead', 'sum'),
).sort_values('mean_lgd', ascending=False)

print("=== LGD by Industry ===")
display(industry_lgd)

### Industry Risk Integration

The industry analysis project provides risk scores (1-5 scale) for each ANZSIC industry division. These scores drive industry-sensitive downturn scalars, recovery haircuts, and margins of conservatism in the enhanced LGD pipeline. See **Notebook 05** for the full integration analysis.

In [ ]:
# Industry risk scores mapped to each loan
from src.industry_risk_integration import (
    IndustryRiskLoader, enrich_loans_with_industry_risk,
    compute_industry_downturn_scalar, INDUSTRY_NAME_MAP,
)

INDUSTRY_DATA_PATH = os.path.join(os.getcwd(), "..", "data", "external", "industry_risk")
loader = IndustryRiskLoader(INDUSTRY_DATA_PATH)
scorecard = loader.load_base_risk_scorecard()

# Show risk score mapping for each industry in portfolio
risk_lookup = loader.get_risk_score_lookup()
industry_lgd["risk_score"] = industry_lgd.index.map(risk_lookup)
industry_lgd["risk_level"] = industry_lgd["risk_score"].apply(
    lambda x: "Elevated" if x > 3.0 else "Medium" if x > 2.5 else "Low"
)

# Industry-adjusted downturn scalars (vs flat 1.15-1.20)
industry_lgd["adjusted_scalar_property"] = compute_industry_downturn_scalar(
    industry_lgd["risk_score"], 1.15
)

print("=== Industry Risk Scores + Adjusted Downturn Scalars ===")
display(industry_lgd[["count", "mean_lgd", "risk_score", "risk_level", "adjusted_scalar_property"]]
        .sort_values("risk_score", ascending=False))

print(f"
Note: Flat Property downturn scalar = 1.15")
print(f"Industry-adjusted range: {industry_lgd["adjusted_scalar_property"].min():.4f} - {industry_lgd["adjusted_scalar_property"].max():.4f}")


## 3. Segmentation

Commercial lending segmentation hierarchy:
```
Level 1: Security type (Property / PPSR / GSR)
  Level 2: Security coverage band
    Level 3: Industry sector (if sufficient data)
```

In [ ]:
engine = CommercialLGDEngine(moc=0.03)

# Primary segmentation: security type × coverage band
lr_lgd = engine.compute_long_run_lgd(loans, segments=['security_type', 'coverage_band'])
print("=== Exposure-Weighted Long-Run LGD ===")
display(lr_lgd)

# Heatmap
pivot = lr_lgd.pivot(index='security_type', columns='coverage_band', values='lgd_long_run')
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Exposure-Weighted Long-Run LGD: Security Type × Coverage Band')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'commercial_segment_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. EAD with Credit Conversion Factors

Commercial facilities often have undrawn components. The CCF converts off-balance-sheet exposure to EAD:

$$\text{EAD} = \text{Drawn} + \text{CCF} \times \text{Undrawn}$$

In [ ]:
# CCF analysis by facility type
ccf_summary = loans.groupby('facility_type').agg(
    count=('loan_id', 'size'),
    mean_ccf=('ccf', 'mean'),
    mean_drawn_pct=('drawn_balance', lambda x: (x / loans.loc[x.index, 'facility_limit']).mean()),
    mean_ead=('ead', 'mean'),
    mean_limit=('facility_limit', 'mean'),
).round(4)

print("=== CCF and EAD by Facility Type ===")
display(ccf_summary)

## 5. Recovery Analysis by PPSR/GSR Security Type

In [ ]:
# Recovery breakdown by cashflow type
recovery_cfs = cashflows[cashflows['cashflow_category'] == 'Recovery']
cost_cfs = cashflows[cashflows['cashflow_category'] == 'Cost']

print("=== Recovery Cashflows by Type ===")
recovery_summary = recovery_cfs.groupby('cashflow_type').agg(
    count=('amount', 'size'),
    total_amount=('amount', 'sum'),
    total_pv=('amount_pv', 'sum'),
    avg_amount=('amount', 'mean'),
).sort_values('total_pv', ascending=False).round(0)
display(recovery_summary)

print("\n=== Cost Cashflows by Type ===")
cost_summary = cost_cfs.groupby('cashflow_type').agg(
    count=('amount', 'size'),
    total_amount=('amount', 'sum'),
    total_pv=('amount_pv', 'sum'),
    avg_amount=('amount', 'mean'),
).sort_values('total_pv', ascending=False).round(0)
display(cost_summary)

In [ ]:
# Recovery rate by security type
loan_recovery = recovery_cfs.groupby('loan_id')['amount_pv'].sum().reset_index()
loan_recovery.columns = ['loan_id', 'total_recovery_pv']
analysis = loans.merge(loan_recovery, on='loan_id', how='left')
analysis['total_recovery_pv'] = analysis['total_recovery_pv'].fillna(0)
analysis['recovery_rate'] = analysis['total_recovery_pv'] / analysis['ead']

fig, ax = plt.subplots(figsize=(10, 5))
analysis.boxplot(column='recovery_rate', by='security_type', ax=ax, rot=20)
ax.set_title('PV Recovery Rate by Security Type')
ax.set_ylabel('PV(Recoveries) / EAD')
plt.suptitle('')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'commercial_recovery_by_security.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6–8. Regulatory Pipeline: Downturn → MoC → Supervisory Floor

In [ ]:
result = engine.apply_overlays(loans)

pipeline_cols = ['realised_lgd', 'lgd_downturn', 'lgd_with_moc', 'lgd_final']

print("=== Regulatory Pipeline Summary ===")
display(result[pipeline_cols].describe().round(4))

print("\n=== Exposure-Weighted Averages ===")
for col in pipeline_cols:
    ewa = exposure_weighted_average(result, lgd_col=col, ead_col='ead')
    print(f"  {col:25s}: {ewa:.4%}")

print("\n=== By Security Type ===")
for sec_type in result['security_type'].unique():
    sub = result[result['security_type'] == sec_type]
    ewa = exposure_weighted_average(sub, 'lgd_final')
    print(f"  {sec_type:25s}: {ewa:.4%} (n={len(sub)}, scalar={sub['downturn_scalar'].iloc[0]:.2f})")

In [ ]:
# Supervisory LGD comparison
print("=== Internal vs Supervisory LGD ===")
for sen in result['seniority'].unique():
    sub = result[result['seniority'] == sen]
    internal = exposure_weighted_average(sub, 'lgd_final')
    supervisory = sub['supervisory_lgd'].iloc[0]
    print(f"  {sen:20s}: Internal={internal:.2%}  Supervisory={supervisory:.0%}  {'OK' if internal >= supervisory * 0.8 else 'REVIEW'}")

## 9. SME Firm-Size Adjustment

In [ ]:
# SME adjustment to correlation (affects RWA, not LGD directly)
result['revenue_millions'] = result['annual_revenue'] / 1e6
result['sme_correlation_adj'] = CommercialLGDEngine.sme_firm_size_adjustment(
    result['revenue_millions']
)

segmented = engine.segment_loans(result)
sme_summary = segmented.groupby('borrower_size').agg(
    count=('loan_id', 'size'),
    mean_revenue=('annual_revenue', 'mean'),
    mean_lgd=('lgd_final', 'mean'),
    mean_sme_adj=('sme_correlation_adj', 'mean'),
).round(4)

print("=== SME Firm-Size Adjustment ===")
print("(Negative adjustment reduces correlation → lower RWA for smaller firms)")
display(sme_summary)

## 10. Validation & Backtesting

In [ ]:
segmented_result = engine.segment_loans(result)
val_report = generate_validation_report(
    segmented_result,
    actual_col='realised_lgd',
    predicted_col='lgd_final',
    segment_col='security_type',
    date_col='default_date'
)

print("=== Accuracy ===")
for k, v in val_report['accuracy'].items():
    print(f"  {k}: {v}")

print("\n=== Conservatism Test ===")
for k, v in val_report['conservatism'].items():
    print(f"  {k}: {v}")

print("\n=== Calibration by Security Type ===")
display(val_report.get('calibration', 'N/A'))

In [ ]:
# Save outputs
result.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'commercial_lgd_results.csv'), index=False)
lr_lgd.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'commercial_segment_summary.csv'), index=False)

print("Outputs saved to outputs/tables/")